# 07-2 문제 해결: 여러 선호 저장·삭제하기 (Store)

`07-2_long_term_memory.ipynb`에서 살펴본 `Store`의 `put()` / `get()` / `search()`에 더해 `delete()`를 새로 사용해본다.  
지금까지는 학습자당 선호를 `"learning_style"` 하나만 저장했는데, 이번에는 **여러 종류의 정보**를 저장하고 그중 하나를 **삭제**할 수 있도록 구조를 확장한다.  

## 요구사항
- `ActionState`에 `key` 필드를 추가해 어떤 종류의 정보인지 지정할 수 있게 한다(예: `"learning_style"`, `"level"`).
- `action`에 `"delete"`를 추가한다. `store.delete(namespace, key)`로 해당 key의 항목을 지운다.
- 같은 key로 `"save"`를 두 번 호출하면 값이 **덮어써져야** 한다(새 항목이 생기는 게 아니라 값만 바뀐다).
- 학습자 A가 `"learning_style"`과 `"level"` 두 가지를 저장하면 `store.search(("preferences", "learner-a"))` 결과가 **2개**여야 한다.
- 그중 `"level"`을 삭제하면 같은 `search()` 결과가 다시 **1개**로 줄어야 한다.
- 저장되지 않은 key를 `"read"`하면 `"저장된 선호 없음"`을 반환해야 한다.
- Node 안에서 `runtime.store`를 쓸 수 있도록, `compile()`에 `checkpointer`와 함께 `store`도 등록해야 한다.

## 힌트
- `namespace = ("preferences", runtime.context.user_id)`는 그대로 두고, 고정 문자열이던 `key = "learning_style"`을 `key = state.key`로 바꾸면 여러 종류를 다룰 수 있다.
- `store.put()`은 같은 namespace+key에 다시 호출하면 기존 값을 덮어쓰는 **upsert**다. 그래서 "저장"과 "값 바꾸기"를 별도 action으로 만들 필요는 없다.
- `store.delete(namespace, key)`는 반환값이 없다. 삭제 후 `result`에 넣을 문자열은 자유롭게 정한다(예: `"삭제 완료"`).
- 없는 key를 `delete`해도 보통 에러가 나지 않는다. 직접 실행해서 확인해 본다.
- `extended_action_node`가 `runtime.store`를 쓰는데, `compile()`에 `store`를 넘기지 않으면 `runtime.store`는 `None`이 된다. `store=store`를 빠뜨리면 어떤 에러가 나는지 직접 확인해본다.

## 기본 설정
`07-2_long_term_memory.ipynb`에서 쓴 것과 같은 `UserContext`, `store`, `checkpointer`를 새로 준비한다. 아래 TODO는 이 셀 다음부터다.

In [1]:
from dataclasses import dataclass
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore

@dataclass
class UserContext:
    user_id: str

store = InMemoryStore()
checkpointer = InMemorySaver()

In [2]:
class ExtendedActionState(BaseModel):
    action: str        # "save", "read", "delete" 중 하나
    # TODO: key 필드를 추가한다. 어떤 종류의 정보인지(예: "learning_style", "level") 지정하는 문자열 필드다.
    key: str
    preference: str
    result: str

def extended_action_node(state: ExtendedActionState, runtime: Runtime[UserContext]) -> dict:
    namespace = ("preferences", runtime.context.user_id)
    key = state.key  # TODO: state에서 key를 가져온다 (고정 문자열 "learning_style"을 쓰지 않는다)

    if state.action == "save":
        # TODO: store.put()으로 저장한다
        runtime.store.put(namespace, key, {"value": state.preference})
        return {"result": "선호 저장 완료"}
    if state.action == "read":
        # TODO: store.get()으로 조회한다
        item = runtime.store.get(namespace, key)
        return {"result": item.value["value"] if item else "저장된 선호 없음"}
    if state.action == "delete":
        # TODO: store.delete(namespace, key)를 호출한다
        runtime.store.delete(namespace, key)
        return {"result": "삭제 완료"}
    raise ValueError("action은 save, read, delete 중 하나여야 한다")

extended_builder = StateGraph(ExtendedActionState, context_schema=UserContext)
extended_builder.add_node("action", extended_action_node)
extended_builder.add_edge(START, "action")
extended_builder.add_edge("action", END)

# TODO: 그래프를 compile한다. Node 안에서 runtime.store를 쓸 수 있으려면 checkpointer 외에 무엇을 더 등록해야 할까?
extended_graph = extended_builder.compile(checkpointer=checkpointer, store = store)

## 자가 점검용 테스트
아래 테스트를 모두 통과하면 완성이다. 통과하지 못하면 `key`를 `state`에서 제대로 가져오는지, `store.put()`/`store.delete()` 인자가 `(namespace, key, value)` / `(namespace, key)` 순서인지 다시 확인한다.

In [3]:
learner_a = UserContext(user_id="learner-a")
thread_a = {"configurable": {"thread_id": "learner-a-thread-1"}}

save_style = extended_graph.invoke(
    {"action": "save", "key": "learning_style", "preference": "실습 코드 예제로 설명해 주세요", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert save_style["result"] == "선호 저장 완료"
assert len(store.search(("preferences", "learner-a"))) == 1

save_level = extended_graph.invoke(
    {"action": "save", "key": "level", "preference": "중급", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert len(store.search(("preferences", "learner-a"))) == 2  # learning_style + level

read_level = extended_graph.invoke(
    {"action": "read", "key": "level", "preference": "", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert read_level["result"] == "중급"

overwrite_level = extended_graph.invoke(
    {"action": "save", "key": "level", "preference": "고급", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert len(store.search(("preferences", "learner-a"))) == 2  # 새 항목이 아니라 덮어쓰기라 개수는 그대로

read_overwritten = extended_graph.invoke(
    {"action": "read", "key": "level", "preference": "", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert read_overwritten["result"] == "고급"

delete_level = extended_graph.invoke(
    {"action": "delete", "key": "level", "preference": "", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert len(store.search(("preferences", "learner-a"))) == 1  # learning_style만 남음

read_after_delete = extended_graph.invoke(
    {"action": "read", "key": "level", "preference": "", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert read_after_delete["result"] == "저장된 선호 없음"

final_style = extended_graph.invoke(
    {"action": "read", "key": "learning_style", "preference": "", "result": ""},
    config=thread_a,
    context=learner_a,
)
assert final_style["result"] == "실습 코드 예제로 설명해 주세요"  # learning_style은 영향 없이 남아 있다

print("모든 테스트 통과")

모든 테스트 통과
